# Görev 5 — Büyük WaveNet ve üç modelin karşılaştırması

Bu notebook **yalnızca Görev 5** için. Kod, Görev 3'teki `build_dataset`, katman sınıfları ve eğitim döngüsünü temel alıyor. `names.txt` dosyasını notebook'la aynı klasöre koyup hücreleri sırayla çalıştırıyorum. Üç modelin eğitimi uzun sürer; video öncesinde sonuçları hazır tutabilirim.

In [1]:
import torch 
import torch.nn.functional as F
import matplotlib.pyplot as plt


In [2]:
words = open('names.txt', 'r', encoding='utf-8').read().splitlines()
print('İsim sayısı:', len(words))

İsim sayısı: 32033


In [3]:
chars = sorted(set(list("".join(words))))
stoi = {s : i+1 for i, s in enumerate(chars)}
stoi["."] = 0
itos = {i : s for s,i in stoi.items()}
vocab_size = len(stoi)

In [4]:
import random
random.seed(42)
random.shuffle(words)

## Aynı isimler, iki bağlam uzunluğu

Önce aynı `build_dataset` fonksiyonuyla bağlamı 3 yapıyorum. Sonra yalnızca `block_size` değerini 8 yapıp aynı isim bölünmelerini yeniden hazırlıyorum.

In [5]:
block_size = 3

def build_dataset(words):
    X, Y = [], []
    for w in words:
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape, Y.shape)
    return X, Y

n1 = int(len(words) * 0.8)
n2 = int(len(words) * 0.9)
Xtr3, Ytr3 = build_dataset(words[:n1])
Xdev3, Ydev3 = build_dataset(words[n1:n2])
Xte3, Yte3 = build_dataset(words[n2:])

block_size = 8
Xtr8, Ytr8 = build_dataset(words[:n1])
Xdev8, Ydev8 = build_dataset(words[n1:n2])
Xte8, Yte8 = build_dataset(words[n2:])

torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])
torch.Size([182625, 8]) torch.Size([182625])
torch.Size([22655, 8]) torch.Size([22655])
torch.Size([22866, 8]) torch.Size([22866])


## Görev 3'teki katmanlar

`BatchNorm1d` burada üç boyutlu veri için düzelttiğim sürüm. Düz MLP'deki `Flatten` sınıfını da ekliyorum.

In [6]:
class Linear:

    def __init__(self, fan_in, fan_out, bias=True):
        self.weight = torch.randn((fan_in, fan_out)) / fan_in**0.5
        self.bias = torch.zeros(fan_out) if bias else None

    def __call__(self, x):
        self.out = x @ self.weight
        if self.bias is not None:
            self.out += self.bias
        return self.out

    def parameters(self):
        return [self.weight] + ([] if self.bias is None else [self.bias])


class BatchNorm1d:

    def __init__(self, dim, eps=1e-5, momentum=0.1):
        self.eps = eps
        self.momentum = momentum
        self.training = True

        # Geri yayılımla öğrenilen parametreler
        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)

        # Eğitim boyunca güncellenen istatistikler
        self.running_mean = torch.zeros(dim)
        self.running_var = torch.ones(dim)

    def __call__(self, x):
        if self.training:
                if x.ndim == 2:
                    dim = 0
                elif x.ndim == 3:
                    dim = (0, 1)
                else:
                    raise ValueError(f"Beklenmeyen girdi şekli: {tuple(x.shape)}")

                xmean = x.mean(dim, keepdim=True)
                xvar = x.var(dim, keepdim=True)
        else:
            xmean = self.running_mean
            xvar = self.running_var

        xhat = (x - xmean) / torch.sqrt(xvar + self.eps)
        self.out = self.gamma * xhat + self.beta

        if self.training:
            with torch.no_grad():
                self.running_mean = (
                    (1 - self.momentum) * self.running_mean
                    + self.momentum * xmean
                )
                self.running_var = (
                    (1 - self.momentum) * self.running_var
                    + self.momentum * xvar
                )

        return self.out

    def parameters(self):
        return [self.gamma, self.beta]


class Tanh:

    def __call__(self, x):
        self.out = torch.tanh(x)
        return self.out

    def parameters(self):
        return []
    # ---------------------------------------------------------------------------
class Embedding:

    def __init__(self, num_embeddings, embedding_dim):
        self.weight = torch.randn((num_embeddings, embedding_dim))

    def __call__(self, IX):
        self.out = self.weight[IX]
        return self.out

    def parameters(self):
        return [self.weight]

class FlattenConsecutive:
  
  def __init__(self, n):
    self.n = n

  def __call__(self, x):
    B, T, C = x.shape #B örnek sayısı, T grup sayısı, C ise her gruptaki sayı sayısı
    x = x.view(B, T//self.n, C*self.n) # , kaç komşu vektörü birleştireceğimi belirliyor. Modelde FlattenConsecutive(2) kullanacağım; yani ikişer ikişer birleştireceğim
    #İkişerli birleştirdiğimde grup sayısı yarıya iner; grup başına sayı iki katına çıkar. Yani (4, 8, 10) şekli (4, 4, 20) olur
    if x.shape[1] == 1:
      x = x.squeeze(1) #Ortadaki 1 artık gereksiz. squeeze onu kaldırıyor
    self.out = x
    return self.out
  
  def parameters(self):
    return []

# class Flatten:

#     def __call__(self, x):
#         self.out = x.view(x.shape[0], -1)
#         return self.out

#     def parameters(self):
#         return []
#Sequential.parameters() ise bütün katmanların parametrelerini dolaşıp tek bir listede topluyor. Böylece eğitim sırasında güncellenecek parametrelere model üzerinden erişiyorum.”
class Sequential: # model(Xb)

    def __init__(self, layers):
        self.layers = layers

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        self.out = x
        return self.out

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

In [7]:
class Flatten:
    def __call__(self, x):
        self.out = x.view(x.shape[0], -1)
        return self.out

    def parameters(self):
        return []

## Eğitim döngüsü

Görev 3'teki 200 bin adımı, `batch_size = 32` değerini ve öğrenme hızı değişimini aynı tutuyorum. Kodun yalnızca model ve veri girişini argüman olarak alan bir fonksiyona taşınmış hali; böylece üç modeli aynı döngüyle çalıştırıyorum.

In [8]:
max_steps = 200000
batch_size = 32

def train_model(model, Xtr, Ytr, Xdev, Ydev):
    parameters = model.parameters()
    n_params = sum(p.nelement() for p in parameters)
    for p in parameters:
        p.requires_grad = True
    lossi = []

    for i in range(max_steps):
        ix = torch.randint(0, Xtr.shape[0], (batch_size,))
        Xb, Yb = Xtr[ix], Ytr[ix]

        logits = model(Xb)
        loss = F.cross_entropy(logits, Yb)
        for p in parameters:
            p.grad = None
        loss.backward()

        lr = 0.1 if i < 150000 else 0.01
        for p in parameters:
            p.data += -lr * p.grad

        if i % 10000 == 0:
            print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
        lossi.append(loss.log10().item())

    for layer in model.layers:
        layer.training = False
    with torch.no_grad():
        dev_loss = F.cross_entropy(model(Xdev), Ydev).item()
    print('Parametre:', n_params, '| Dev loss:', round(dev_loss, 6))
    return n_params, dev_loss, lossi

## 1. Bağlam 3 düz MLP

Önce eski düz MLP'yi `n_embd = 10` ve `n_hidden = 200` ile eğitiyorum.

In [9]:
torch.manual_seed(42)
n_embd = 10
n_hidden = 200
model_3 = Sequential([
    Embedding(vocab_size, n_embd),
    Flatten(),
    Linear(n_embd * 3, n_hidden, bias=False),
    BatchNorm1d(n_hidden),
    Tanh(),
    Linear(n_hidden, vocab_size),
])
with torch.no_grad():
    model_3.layers[-1].weight *= 0.1

params_3, dev_3, loss_3 = train_model(model_3, Xtr3, Ytr3, Xdev3, Ydev3)

      0/ 200000: 3.2966
  10000/ 200000: 2.2322
  20000/ 200000: 2.4111
  30000/ 200000: 2.1004
  40000/ 200000: 2.3157
  50000/ 200000: 2.2104
  60000/ 200000: 1.9653
  70000/ 200000: 1.9767
  80000/ 200000: 2.6738
  90000/ 200000: 2.0837
 100000/ 200000: 2.2730
 110000/ 200000: 1.7491
 120000/ 200000: 2.2891
 130000/ 200000: 2.3443
 140000/ 200000: 2.1731
 150000/ 200000: 1.8246
 160000/ 200000: 1.7614
 170000/ 200000: 2.2419
 180000/ 200000: 2.0803
 190000/ 200000: 2.1326
Parametre: 12097 | Dev loss: 2.106529


## 2. Bağlam 8 düz MLP

Aynı düz modeli bu kez 8 karakterle eğitiyorum. İlk `Linear` katmanı bu yüzden `n_embd * 8` girdi alıyor.

In [10]:
torch.manual_seed(42)
n_embd = 10
n_hidden = 200
model_8 = Sequential([
    Embedding(vocab_size, n_embd),
    Flatten(),
    Linear(n_embd * 8, n_hidden, bias=False),
    BatchNorm1d(n_hidden),
    Tanh(),
    Linear(n_hidden, vocab_size),
])
with torch.no_grad():
    model_8.layers[-1].weight *= 0.1

params_8, dev_8, loss_8 = train_model(model_8, Xtr8, Ytr8, Xdev8, Ydev8)

      0/ 200000: 3.2847
  10000/ 200000: 2.0647
  20000/ 200000: 1.9722
  30000/ 200000: 2.0948
  40000/ 200000: 1.9738
  50000/ 200000: 2.1287
  60000/ 200000: 2.3574
  70000/ 200000: 1.9131
  80000/ 200000: 2.0735
  90000/ 200000: 2.0968
 100000/ 200000: 1.4963
 110000/ 200000: 2.1294
 120000/ 200000: 2.2324
 130000/ 200000: 2.2071
 140000/ 200000: 2.2326
 150000/ 200000: 1.8908
 160000/ 200000: 1.6867
 170000/ 200000: 2.0968
 180000/ 200000: 1.7824
 190000/ 200000: 1.9151
Parametre: 22097 | Dev loss: 2.034248


## 3. Bağlam 8 büyük WaveNet

Görev 3'ün `8 → 4 → 2 → 1` mimarisi aynı. Yalnızca `n_embd = 24` ve `n_hidden = 128` yaparak modeli büyütüyorum.

In [11]:
torch.manual_seed(42)
n_embd = 24
n_hidden = 128
model_wave = Sequential([
    Embedding(vocab_size, n_embd),
    FlattenConsecutive(2), Linear(n_embd * 2, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
    FlattenConsecutive(2), Linear(n_hidden * 2, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
    FlattenConsecutive(2), Linear(n_hidden * 2, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
    Linear(n_hidden, vocab_size),
])
with torch.no_grad():
    model_wave.layers[-1].weight *= 0.1

params_wave, dev_wave, loss_wave = train_model(model_wave, Xtr8, Ytr8, Xdev8, Ydev8)

      0/ 200000: 3.3167
  10000/ 200000: 2.0576
  20000/ 200000: 2.0723
  30000/ 200000: 2.5134
  40000/ 200000: 2.1476
  50000/ 200000: 1.7836
  60000/ 200000: 2.2592
  70000/ 200000: 1.9331
  80000/ 200000: 1.6875
  90000/ 200000: 2.0395
 100000/ 200000: 1.7736
 110000/ 200000: 1.9569
 120000/ 200000: 1.7465
 130000/ 200000: 1.8126
 140000/ 200000: 1.7406
 150000/ 200000: 1.7466
 160000/ 200000: 1.8806
 170000/ 200000: 1.6266
 180000/ 200000: 1.6476
 190000/ 200000: 1.8555
Parametre: 76579 | Dev loss: 1.993652


## Tek tablo: parametre sayısı ve dev loss

İlk iki satır aynı düz MLP'de bağlamı 3'ten 8'e çıkarmanın sonucunu, son satır ise büyütülmüş WaveNet'in sonucunu gösterir. Dev loss değerleri eğitimin sonunda hesaplanır.

In [12]:
print(f"{'Model':<23} {'Parametre sayısı':>17} {'Dev loss':>12}")
print('-' * 54)
for name, count, dev in [
    ('Bağlam 3 MLP', params_3, dev_3),
    ('Bağlam 8 düz MLP', params_8, dev_8),
    ('Bağlam 8 WaveNet', params_wave, dev_wave),
]:
    print(f'{name:<23} {count:>17,d} {dev:>12.6f}')

Model                    Parametre sayısı     Dev loss
------------------------------------------------------
Bağlam 3 MLP                       12,097     2.106529
Bağlam 8 düz MLP                   22,097     2.034248
Bağlam 8 WaveNet                   76,579     1.993652
